# Unit 5 hands-on: fine-tune Whisper for speech recognition (MINDS-14)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohammadDaleen/Audio-Course/blob/main/units/unit5_automatic_speech_recognition/colab_handson.ipynb)

This notebook walks you through the Hugging Face Audio Course **Unit 5 hands-on** from start to finish.

**What you'll do:** take `whisper-tiny` — a small model that already understands speech — and
*fine-tune* it on **MINDS-14**, a set of recorded banking requests, so it transcribes that kind of
audio better. Then you'll push your trained model to the Hugging Face Hub.

**What is "fine-tuning"?** Training a speech model from scratch needs hundreds of thousands of hours
of audio. Instead we start from a model that has already learned what speech sounds like, and nudge
its weights with a few hundred examples of *our* audio. It's fast and it works.

**How you're graded:** on **WER** (Word Error Rate) — the fraction of reference words the model got
wrong. You pass if the **normalised WER is below 0.37**.

> ⚠️ Report WER as a **fraction**, not a percentage. 0.42 — not 42. This trips people up every year.

For calibration, `whisper-tiny` scores about **0.44** on this data *before* any training. So the base
model does not pass, but you are not far off — a few hundred training steps is enough.

**Before you run anything:**
1. Set the runtime to a GPU: **Runtime → Change runtime type → T4 GPU → Save**. Training on CPU would
   take hours; on a T4 it is about 25 minutes.
2. Run the cells top to bottom (Shift+Enter on each, or **Runtime → Run all**).

## Step 1 — Install the libraries

We pin versions so this notebook keeps working:

- **`datasets==3.6.0`** — version 4 changed the audio backend to `torchcodec` + FFmpeg. 3.x decodes
  audio with `soundfile`/`librosa`, which is what the rest of this code expects.
- **`transformers>=4.46`** — needed for the current training API (`eval_strategy`,
  `processing_class`) used below.
- **`jiwer`** — the edit-distance library that actually computes WER. `evaluate.load("wer")` imports
  it under the hood and fails without it.
- `evaluate` (the metric wrapper), `accelerate` (runs training on the GPU), `soundfile`/`librosa`
  (decode the audio), `tensorboard` (training charts).

We do **not** install PyTorch — Colab already ships a GPU build of it.

In [ ]:
!pip install -q "transformers>=4.46,<5" "datasets==3.6.0" "evaluate>=0.4" "jiwer>=3.0" \
                "accelerate>=0.30" "soundfile>=0.12.1" "librosa>=0.10" tensorboard

> If a later cell raises a strange import error, do **Runtime → Restart session**, then run again
> from Step 2 (you don't need to re-run Step 1 — the packages are already installed).

## Step 2 — Check the GPU and log in to Hugging Face

The first lines confirm a GPU is active. The login stores a token so the notebook can upload your
model at the end.

**Use a WRITE token — this is the #1 thing people get wrong.** A read-only token cannot create your
model repository and you'll get a `403 Forbidden` error later. Create one here:
**https://huggingface.co/settings/tokens → Create new token → type "Write" → Create → copy it.**

Run the cell, then paste the **write** token into the box that appears.

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available(),
      "-", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - set Runtime -> T4 GPU!")

from huggingface_hub import notebook_login
notebook_login()   # paste a WRITE token

Now confirm the login worked and the token can actually write. You want your own username and a role
of `write` (or a fine-grained token allowed to create repos):

In [ ]:
from huggingface_hub import whoami
info = whoami()
print("logged in as:", info["name"])
print("token role  :", info.get("auth", {}).get("accessToken", {}).get("role", "(fine-grained)"))

## Step 3 — Load MINDS-14 and split it

**MINDS-14** is a set of short recordings of people making banking requests ("I'd like to deposit a
cheque"), with a human transcription for each. We use the **American English** subset, `en-US`.

The hands-on specifies the split exactly: **the first 450 examples for training, the rest for
evaluation**. The `en-US` config has 563 rows, so that gives 450 train / 113 test.

Two things worth understanding:

- **`select_columns` + `rename_column`** — MINDS-14 ships extra columns we don't need, and it calls
  the text column `transcription`. We keep just the audio and the text, and rename it to `sentence`.
- **`cast_column(..., Audio(sampling_rate=16_000))`** — MINDS-14 is **8 kHz telephone audio** and
  Whisper needs **16 kHz**. This line makes `datasets` resample on the fly. Skip it and you get an
  `ImportError: torchaudio is required to resample` later.

The `min(450, 80%)` guard just means the notebook won't crash with `IndexError` if you switch to a
smaller language config.

In [ ]:
from datasets import load_dataset, Audio, DatasetDict

minds = load_dataset("PolyAI/minds14", "en-US", split="train")
print(minds)

minds = minds.select_columns(["audio", "transcription"]).rename_column("transcription", "sentence")
minds = minds.cast_column("audio", Audio(sampling_rate=16_000))

n_train = min(450, int(len(minds) * 0.8))          # the hands-on says the first 450
dataset = DatasetDict(
    train=minds.select(range(n_train)),
    test=minds.select(range(n_train, len(minds))),
)
print(dataset)
print("\nexample:", dataset["train"][0]["sentence"])

> Using a different language? `en-AU` and `en-GB` are drop-in replacements for `"en-US"` above.
> Whichever you pick, keep the same 450/rest split.

## Step 4 — The processor (and one bug you must work around)

A `WhisperProcessor` bundles two things:

- a **feature extractor**, which turns raw audio into the log-mel spectrogram Whisper's encoder eats,
- a **tokenizer**, which turns text into token ids (the *labels* the model learns to produce).

Whisper is multilingual, so every label sequence starts with special tokens saying which language
this is and what job to do: `<|startoftranscript|><|en|><|transcribe|><|notimestamps|>`.

**The bug:** on current `transformers`, passing `language=` and `task=` to `from_pretrained()` does
*not* reach the fast tokenizer. It quietly encodes labels as `<|startoftranscript|><|notimestamps|>`
with `<|en|><|transcribe|>` **missing** — while generation at inference time *does* add them. You'd
train on one format and decode with another, and your WER would be worse for no visible reason.

`set_prefix_tokens(...)` fixes it. The cell prints the prefix so you can see it's right.

In [ ]:
from transformers import WhisperProcessor

MODEL_ID = "openai/whisper-tiny"
LANGUAGE, TASK = "english", "transcribe"

processor = WhisperProcessor.from_pretrained(MODEL_ID, language=LANGUAGE, task=TASK)
processor.tokenizer.set_prefix_tokens(language=LANGUAGE, task=TASK)   # <- the fix

prefix = processor.tokenizer.prefix_tokens
print("label prefix:", processor.tokenizer.convert_ids_to_tokens(prefix))
# you should see: ['<|startoftranscript|>', '<|en|>', '<|transcribe|>', '<|notimestamps|>']

## Step 5 — Turn audio and text into model inputs

`prepare_dataset` runs on every example and produces:

- `input_features` — the log-mel spectrogram, always shaped `(80, 3000)` because Whisper pads or
  truncates everything to exactly **30 seconds**,
- `labels` — the tokenised transcription,
- `input_length` — how long the clip is, which we use to drop anything over 30 s (it would be
  silently truncated, so its label would mention words that are no longer audible).

**`num_proc=1` is required by the hands-on.** Multiprocessing here is a common source of hangs, and
the task explicitly asks for it.

In [ ]:
def prepare_dataset(example):
    audio = example["audio"]
    out = processor(
        audio=audio["array"],
        sampling_rate=audio["sampling_rate"],
        text=example["sentence"],
    )
    # NOTE: do NOT write out["labels"][0] here. For a single string the tokenizer
    # returns a flat list of ids, so [0] would give you one integer and the collator
    # in Step 6 would crash.
    out["input_length"] = len(audio["array"]) / audio["sampling_rate"]
    return out

dataset = dataset.map(
    prepare_dataset,
    remove_columns=dataset.column_names["train"],
    num_proc=1,                      # required by the hands-on
)
dataset = dataset.filter(lambda length: length < 30.0, input_columns=["input_length"])
dataset = dataset.remove_columns(["input_length"])
print(dataset)

## Step 6 — The data collator (how a batch gets built)

A batch has to be one rectangular tensor, but our examples aren't the same size. Audio and labels
need **different** treatment:

- **Audio is already uniform** — every clip became a `(80, 3000)` block in Step 5, so there's nothing
  to pad. We just stack them.
- **Labels are ragged** — one transcription might be 12 tokens and another 40. We pad the short ones
  up to the longest in the batch.

Then two subtle steps:

- **Padding is replaced with `-100`.** PyTorch's loss function (`cross_entropy`) has an
  `ignore_index` argument that defaults to `-100`, so those positions contribute nothing. Without
  this the model would be rewarded for predicting padding — the most common silent bug in this whole
  pipeline.
- **A leading start token is stripped.** The model prepends `<|startoftranscript|>` itself when it
  shifts the labels right internally, so leaving ours in would teach it to emit two.

In [ ]:
from dataclasses import dataclass
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: object
    decoder_start_token_id: int

    def __call__(self, features):
        # 1. audio: already all (80, 3000), just stack
        input_features = [{"input_features": f["input_features"][0]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 2. labels: pad to the longest in this batch
        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # 3. padding -> -100 so it contributes no loss
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # 4. drop a duplicated start-of-transcript token
        if (labels[:, 0] == self.decoder_start_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

## Step 7 — The metric you're graded on

We report two numbers:

- **`wer_ortho`** (orthographic) — compare the strings exactly as decoded, including capital letters
  and punctuation.
- **`wer`** (normalised) — lowercase both sides and strip punctuation first. **This is the one that
  must be below 0.37.**

Why both? Whisper writes `"I'd like to deposit a cheque."` while the reference might say
`"i would like to deposit a cheque"`. Orthographic WER punishes every one of those cosmetic
differences; normalised WER measures whether it heard the right words.

The last few lines drop any example whose reference normalises to an empty string — WER divides by
the number of reference words, so an empty reference would divide by zero.

> Return **fractions**. The course's own chapter multiplies by 100; the hands-on explicitly asks you
> not to.

In [ ]:
import evaluate
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

metric = evaluate.load("wer")
normalizer = BasicTextNormalizer()

def compute_metrics(pred):
    pred_ids, label_ids = pred.predictions, pred.label_ids

    # put the real pad token back where we wrote -100, so this can be decoded
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)

    wer_ortho = metric.compute(predictions=pred_str, references=label_str)

    pred_norm = [normalizer(p) for p in pred_str]
    label_norm = [normalizer(l) for l in label_str]
    keep = [i for i in range(len(label_norm)) if len(label_norm[i]) > 0]
    wer = metric.compute(
        predictions=[pred_norm[i] for i in keep],
        references=[label_norm[i] for i in keep],
    )

    return {"wer_ortho": wer_ortho, "wer": wer}   # fractions, NOT percentages

## Step 8 — Load the model and tell it what language to speak

Three settings matter here:

- **`model.config.use_cache = False`** — gradient checkpointing (Step 9) saves memory by recomputing
  activations instead of storing them, and that's incompatible with the attention cache.
- **`model.generation_config.use_cache = True`** — but evaluation *generates* text, and there we do
  want the cache, or every eval pass runs about twice as slow.
- **`generation_config.language` / `.task`** — this is what makes the model transcribe English. Older
  tutorials monkey-patch `model.generate` with `functools.partial` instead; don't. A patched
  `generate` is not saved by `push_to_hub`, so your uploaded model would forget its language.

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(MODEL_ID)

model.config.use_cache = False              # incompatible with gradient checkpointing
model.generation_config.use_cache = True    # but generation during eval wants it
model.generation_config.language = LANGUAGE
model.generation_config.task = TASK
model.generation_config.forced_decoder_ids = None   # clears a legacy setting in the config

collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    decoder_start_token_id=model.config.decoder_start_token_id,
)

## Step 9 — Training configuration

The knobs, in plain terms:

- **`max_steps=600`** — how many batches the model trains on. With batch size 16 over 450 examples
  that's roughly 21 passes through the data. Raise it to 1000 if you don't clear 0.37.
- **`learning_rate=1e-5`** — how big each update is. Fine-tuning wants a *small* rate; too high and
  you destroy what the model already knows.
- **`lr_scheduler_type="constant_with_warmup"` + `warmup_steps=50`** — start tiny and ramp up over the
  first 50 steps, which stops early batches from wrecking the weights.
- **`per_device_train_batch_size=16`** — clips processed at once. Halve it if you hit
  `CUDA out of memory`, and double `gradient_accumulation_steps` to compensate.
- **`gradient_checkpointing=True`** and **`fp16=True`** — memory and speed tricks that let a T4 handle
  this comfortably. (GPU only.)
- **`predict_with_generate=True`** — during eval, actually *generate* text rather than just measuring
  loss. WER needs real transcriptions.
- **`eval_strategy` / `save_strategy="steps"` with `eval_steps=save_steps=50`** — check WER every 50
  steps. These two must match for the next line to work. WER on a 113-clip test set is noisy and
  doesn't fall smoothly, so checking often gives the next setting more chances to catch a good one.
- **`load_best_model_at_end=True`** with **`metric_for_best_model="wer"`** and
  **`greater_is_better=False`** — keep the checkpoint with the *lowest* WER, not the last one. This is
  the safety net that protects your 0.37: even if late steps overfit, you keep the best result.
- **`push_to_hub=True`** — upload to your Hub account.

> Note `eval_strategy`, not `evaluation_strategy`. The old name was removed and raises `TypeError`.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

REPO_NAME = "whisper-tiny-finetuned-minds14-en"   # becomes your Hub repo name

training_args = Seq2SeqTrainingArguments(
    output_dir=REPO_NAME,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=1,
    learning_rate=1e-5,
    lr_scheduler_type="constant_with_warmup",
    warmup_steps=50,
    max_steps=600,
    gradient_checkpointing=True,
    fp16=True,
    fp16_full_eval=True,
    eval_strategy="steps",          # NOT evaluation_strategy (removed)
    save_strategy="steps",          # must match eval_strategy
    eval_steps=50,                  # WER is noisy; check often so the best one gets kept
    save_steps=50,
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=225,
    logging_steps=25,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,        # lower WER is better
    push_to_hub=True,
    report_to=["tensorboard"],
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=collator,
    compute_metrics=compute_metrics,
    processing_class=processor,     # current API (older tutorials used tokenizer=)
)

> **If this cell raises `403 Forbidden ... create`**, your token is read-only. Re-run Step 2 with a
> **Write** token, then run this cell again. (The repo is created the moment `push_to_hub=True` is
> set — before any training — so you lose no time fixing it.)

## Step 10 — Measure the baseline first

Worth 90 seconds: score the model *before* training, so you can see what fine-tuning bought you.
Expect `wer` around **0.44** — above the 0.37 threshold, which is why this exercise exists.

In [ ]:
baseline = trainer.evaluate()
print("baseline wer_ortho:", round(baseline["eval_wer_ortho"], 4))
print("baseline wer      :", round(baseline["eval_wer"], 4), "   (target: < 0.37)")

## Step 11 — Train (this is the ~25 minute step)

Watch the table: every 100 steps you'll see `Wer Ortho` and `Wer`. You want `Wer` to fall below
**0.37**. Keep the Colab tab open so the runtime doesn't disconnect.

In [ ]:
trainer.train()

## Step 12 — Did you pass?

Because of `load_best_model_at_end`, `trainer` is now holding the best checkpoint, not the last one.

In [ ]:
metrics = trainer.evaluate()
print("wer_ortho :", round(metrics["eval_wer_ortho"], 4))
print("wer       :", round(metrics["eval_wer"], 4))
print()
print("PASS 🎉" if metrics["eval_wer"] < 0.37 else "NOT YET - see the Troubleshooting cell at the bottom")

## Step 13 — Push to the Hub (this is what counts for the certificate)

This uploads your model plus a model card. The three `kwargs` the hands-on requires are
`dataset_tags`, `finetuned_from` and `tasks` — they're the tags the course looks for. The others just
make your model card nicer.

In [ ]:
kwargs = {
    "dataset_tags": "PolyAI/minds14",     # required
    "finetuned_from": MODEL_ID,           # required
    "tasks": "automatic-speech-recognition",  # required
    "dataset": "MINDS-14 (en-US)",
    "language": "en",
    "model_name": REPO_NAME,
}
trainer.push_to_hub(**kwargs)

Your model is now at `https://huggingface.co/<your-username>/whisper-tiny-finetuned-minds14-en`. 🎉

## Step 14 (optional) — Hear the difference

Load your uploaded model as a ready-to-use pipeline and compare it against the original
`whisper-tiny` on the same clip.

In [ ]:
from transformers import pipeline

repo = f"{whoami()['name']}/{REPO_NAME}"
tuned = pipeline("automatic-speech-recognition", model=repo, device=0)
base = pipeline("automatic-speech-recognition", model=MODEL_ID, device=0)

sample = minds[n_train]           # first example the model never trained on
gen = {"task": "transcribe", "language": "english"}

print("reference  :", sample["sentence"])
print("before     :", base(sample["audio"]["array"], generate_kwargs=gen)["text"].strip())
print("fine-tuned :", tuned(sample["audio"]["array"], generate_kwargs=gen)["text"].strip())

## Step 15 (optional) — A quick demo

`debug=True` keeps the cell running so the widget stays interactive inside Colab.

In [ ]:
!pip install -q gradio
import gradio as gr, librosa

def transcribe_speech(filepath):
    if filepath is None:
        return "Record or upload some audio first."
    array, _ = librosa.load(filepath, sr=16_000, mono=True)
    out = tuned({"array": array, "sampling_rate": 16_000},
                chunk_length_s=30, batch_size=8, ignore_warning=True,
                generate_kwargs={"task": "transcribe", "language": "english"})
    return out["text"].strip()

mic = gr.Interface(fn=transcribe_speech,
                   inputs=gr.Audio(sources="microphone", type="filepath"),
                   outputs=gr.Textbox(label="Transcription"))
upload = gr.Interface(fn=transcribe_speech,
                      inputs=gr.Audio(sources="upload", type="filepath"),
                      outputs=gr.Textbox(label="Transcription"))

gr.TabbedInterface([mic, upload], ["Microphone", "Audio file"]).launch(debug=True)

## Troubleshooting

- **`403 Forbidden ... create`** — your token is read-only. Make a **Write** token at
  https://huggingface.co/settings/tokens, re-run Step 2, then continue.
- **"NO GPU" printed in Step 2** — set **Runtime → Change runtime type → T4 GPU** and re-run.
- **`TypeError: ... unexpected keyword argument 'evaluation_strategy'`** — it's `eval_strategy` now.
- **`TypeError: ... unexpected keyword argument 'tokenizer'`** on `Seq2SeqTrainer`** — it's
  `processing_class=processor`.
- **`KeyError: 'sentence'`** — you skipped the `rename_column` in Step 3; MINDS-14 calls the column
  `transcription`.
- **`ImportError: torchaudio is required to resample`** — you skipped the
  `cast_column("audio", Audio(sampling_rate=16_000))` in Step 3.
- **`ImportError` from `evaluate.load("wer")`** — `jiwer` didn't install. Re-run Step 1.
- **The `.map` in Step 5 hangs** — you changed `num_proc`. Put it back to `1`.
- **Decoding produces gibberish / an out-of-range id error** — you removed the
  `label_ids[label_ids == -100] = pad_token_id` line in Step 7.
- **`CUDA out of memory`** — set `per_device_train_batch_size=8` and
  `gradient_accumulation_steps=2` in Step 9.
- **WER reads `44` instead of `0.44`** — you multiplied by 100 somewhere. The threshold is a fraction.
- **WER stuck above 0.37** — in order: raise `max_steps` to 1000; then try `learning_rate=6e-6`; then
  shuffle before splitting (`minds.shuffle(seed=42)` in Step 3 — MINDS-14 rows are grouped by intent,
  so the untouched tail is a slightly biased eval set); then switch `MODEL_ID` to
  `"openai/whisper-base"`, which is still comfortable on a T4 (remember it also changes
  `finetuned_from`).
- **Runtime disconnected mid-training** — Colab's free tier disconnects when idle. Keep the tab open
  and visible, or lower `max_steps`.